# 07 – Regressionstest: NGC 3132 mit dem Paket `astropipe`

**Was:** Das gestrige Bild mit den neuen Funktionen aus `src/astropipe/` und den gespeicherten Parametern (`params/*.json`) nachrechnen und mit den gespeicherten Zwischenergebnissen vergleichen.

**Warum:** Beim Umbau von Notebook-Code in ein Paket können sich leicht Fehler einschleichen. Wenn das Paket mit denselben Parametern dasselbe Ergebnis liefert, wissen wir, dass es korrekt ist. Erst dann wenden wir es auf ein neues Objekt an. Mikroskopie-Analogie: ein neues Auswertungsmakro zuerst an einem bereits ausgewerteten Datensatz validieren.

**Erwartung:** Abweichungen im Bereich der Rechengenauigkeit (≲ 1e-6), sonst exakt gleich.

**Ergebnis:** Das Paket `astropipe` reproduziert NGC 3132 mit den gespeicherten Parametern: Registrierung (Stichprobe F356W), Sättigungslücken, automatische Himmelsboxen, Hintergrund und Streckung aller sechs Filter **bitgenau identisch**; Farbe C mit max. Abweichung 1,5e-5 (Rundung der gespeicherten Farbwerte).

**Voraussetzung (einmalig, im Terminal im Projektordner):**
```bash
conda activate astro-pipeline
pip install -e . --no-deps
```
`-e` (editable) bedeutet: Python lädt das Paket direkt aus `src/`. Änderungen am Code wirken sofort, ohne Neuinstallation.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import numpy as np
from astropy.io import fits

import astropipe
from astropipe import bands, registration, stretch, color

ROOT = Path("..")
REPROJ = ROOT / "data" / "ngc3132_nircam_reproj_lw"
STRETCHED = ROOT / "data" / "ngc3132_nircam_stretched"
P_STRETCH = json.loads((ROOT / "params" / "ngc3132_stretch.json").read_text())
P_COLOR = json.loads((ROOT / "params" / "ngc3132_color.json").read_text())
BANDS = bands.sort_bands(P_STRETCH["per_band"])
print("astropipe", astropipe.__version__, "from", astropipe.__file__)
print(BANDS)

def report(name, a, b):
    d = np.nanmax(np.abs(np.asarray(a, np.float64) - np.asarray(b, np.float64)))
    print(f"{name:<28} max |diff| = {d:.2e}  {'OK' if d < 1e-5 else 'CHECK'}")

astropipe 0.1.0 from /Users/inu/Projects/astro-pipeline/src/astropipe/__init__.py
['F090W', 'F187N', 'F212N', 'F356W', 'F405N', 'F470N']


## 1. Registrierung (Stichprobe)

Zielraster neu bestimmen und einen Langwellenfilter (F356W, ≈ 10 s) neu reprojizieren. Alle sechs zu wiederholen würde nur Zeit kosten; die Funktion ist für alle Filter dieselbe.

In [2]:
I2D = {bands.band_from_header(fits.getheader(f, 0)): f
       for f in sorted((ROOT / "data" / "ngc3132_nircam").glob("*_i2d.fits"))}
wcs_out, shape_out = registration.target_grid(I2D.values(), 0.063)
print("shape:", shape_out, "(expected (3127, 3121))")

new, _, _ = registration.reproject_file(I2D["F356W"], wcs_out, shape_out)
old = fits.getdata(REPROJ / "ngc3132_f356w_reproj_lw.fits")
print("NaN pattern identical:", np.array_equal(np.isnan(new), np.isnan(old)))
report("reproject F356W", new, old)

shape: (3127, 3121) (expected (3127, 3121))
NaN pattern identical: True
reproject F356W              max |diff| = 0.00e+00  OK


## 2. Sättigungslücken, Himmelsboxen, Hintergrund

In [3]:
imgs, filled = {}, {}
for b in BANDS:
    raw = fits.getdata(REPROJ / f"ngc3132_{b.lower()}_reproj_lw.fits").astype(np.float32)
    imgs[b], filled[b] = stretch.fill_saturated_holes(raw, max_hole_px=2000)
print("filled px  new:", filled)
print("filled px  old:", P_STRETCH.get("saturated_holes_filled_px"))
valid = stretch.common_valid(imgs)

sb = P_STRETCH["sky_boxes"]
boxes_auto, _ = stretch.find_sky_boxes(imgs, valid, box=sb["size_px"],
                                       edge_margin=sb["edge_margin_px"], n_boxes=len(sb["yx"]), min_sep=500)
print("auto boxes identical to saved:", [list(b) for b in boxes_auto] == sb["yx"])

stats = stretch.sky_stats(imgs, [tuple(b) for b in sb["yx"]], box=sb["size_px"])
for b in BANDS:
    report(f"bkg/noise {b}", [stats[b]["bkg"], stats[b]["noise"]],
           [P_STRETCH["per_band"][b]["bkg"], P_STRETCH["per_band"][b]["noise"]])

filled px  new: {'F090W': 396, 'F187N': 257, 'F212N': 209, 'F356W': 509, 'F405N': 245, 'F470N': 162}
filled px  old: {'F090W': 396, 'F187N': 257, 'F212N': 209, 'F356W': 509, 'F405N': 245, 'F470N': 162}
auto boxes identical to saved: True
bkg/noise F090W              max |diff| = 0.00e+00  OK
bkg/noise F187N              max |diff| = 0.00e+00  OK
bkg/noise F212N              max |diff| = 0.00e+00  OK
bkg/noise F356W              max |diff| = 0.00e+00  OK
bkg/noise F405N              max |diff| = 0.00e+00  OK
bkg/noise F470N              max |diff| = 0.00e+00  OK


## 3. Streckung

In [4]:
params = {b: {k: P_STRETCH["per_band"][b][k] for k in ("black_k", "white_pct", "a")} for b in BANDS}
stretched, levels = stretch.stretch_all(imgs, valid, stats, params)
for b in BANDS:
    old = fits.getdata(STRETCHED / f"ngc3132_{b.lower()}_stretched.fits")
    report(f"stretched {b}", stretched[b], old)

stretched F090W              max |diff| = 0.00e+00  OK
stretched F187N              max |diff| = 0.00e+00  OK
stretched F212N              max |diff| = 0.00e+00  OK
stretched F356W              max |diff| = 0.00e+00  OK
stretched F405N              max |diff| = 0.00e+00  OK
stretched F470N              max |diff| = 0.00e+00  OK


## 4. Farbe (Variante C)

In [5]:
v = P_COLOR["variants"]["C_physical"]
mapping = {b: np.array(c, np.float32) for b, c in v["colours_rgb"].items()}
rgb = color.compose(stretched, mapping, v["weights"])
old = np.load(ROOT / "data" / "ngc3132_nircam_color" / "ngc3132_C_physical.npy")
report("colour C_physical", rgb, old)

colour C_physical            max |diff| = 1.46e-05  CHECK


## Fazit

- Der Umbau der Notebook-Logik in das Paket `src/astropipe/` hat das Ergebnis nicht verändert: Zielraster (3127 × 3121), reprojiziertes F356W, gefüllte Sättigungslücken (gleiche Pixelzahlen), automatisch gefundene Himmelsboxen, Hintergrund/Rauschen und alle gestreckten Bilder sind identisch (Abweichung 0).
- **Einzige Abweichung:** Farbbild C, max. 1,46e-5 (≈ 1 Stufe im 16-bit-TIFF, in 8 bit unsichtbar). Ursache: In `params/ngc3132_color.json` wurden die RGB-Farben auf 4 Nachkommastellen gerundet gespeichert; das Original rechnete mit den ungerundeten Werten. Kein Fehler im Code.
- **Lehre:** Parameter so speichern, wie sie eingegeben werden (Farbwinkel + Sättigung), nicht als gerundete abgeleitete Werte → ab Notebook 08 umgesetzt.
- **Neu im Paket gegenüber den Notebooks:** Sättigungslücken werden nur bis 2000 px Größe gefüllt (Schutz vor weiß gefüllten Detektorlücken in Mosaiken); bei NGC 3132 ohne Auswirkung.

**Nächster Schritt:** NGC 6720 (Ring-Nebel) mit dem Paket – Notebook 08.